# TimeXer AB DSIO Test

이 노트북은 `tb_master_exo.parquet`를 기준으로 `TimeXer`를 학습하는 수동 점검용 notebook입니다.

## 이 notebook에서 하는 일
1. repo 경로와 import 환경을 정리합니다.
2. `tb_master_exo.parquet`를 주 데이터 소스로 읽습니다.
3. `tb_master_target.parquet`의 `demand_qty`를 join해서 TimeXer 학습용 one-table long data를 만듭니다.
4. dataloader batch shape를 먼저 확인합니다.
5. `timexer_base`를 학습하고 checkpoint / manifest 경로를 확인합니다.

## TimeXer v1 전제
- 현재 라이브러리의 TimeXer는 `past continuous exogenous`만 사용합니다.
- `future_exo_cont_cols`는 비워둡니다.
- 공식 구현에 맞춰 `lookback % patch_len == 0` 조건을 지켜야 합니다.
- `tb_master_exo.parquet`에는 target이 없으므로 `tb_master_target.parquet`에서 `demand_qty`를 join합니다.


In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path
from pprint import pprint

import numpy as np
import polars as pl
import torch


# Remote kernel이면 repo 절대경로를 직접 넣어도 됩니다.
# Example: REPO_ROOT_OVERRIDE = Path("/home/ubuntu/ts_forecaster_lib")
REPO_ROOT_OVERRIDE = None


def find_repo_root(start: Path, override: Path | None = None) -> Path:
    candidates = []
    if override is not None:
        candidates.append(Path(override).expanduser().resolve())

    env_repo_root = os.environ.get("TS_FORECASTER_REPO_ROOT")
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())

    candidates.extend([start, *start.parents])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate

    raise RuntimeError(
        "Could not locate repo root. Set REPO_ROOT_OVERRIDE or TS_FORECASTER_REPO_ROOT first."
    )


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR, override=REPO_ROOT_OVERRIDE)
SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from modeling_module import (
    ArchitectureConfig,
    ArtifactConfig,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,
    RuntimeConfig,
    TimexerArchitectureConfig,
    TrainRequest,
    TrainerConfig,
    build_dataloader,
    train,
)
from modeling_module.utils.device import select_default_device


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEFAULT_DEVICE, DEVICE_NOTE = select_default_device()

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)
print("PYTHON   :", sys.executable)
print("TORCH    :", torch.__version__)
print("DEVICE   :", DEFAULT_DEVICE)
if DEVICE_NOTE:
    print("DEVICE_NOTE:", DEVICE_NOTE)


## 1. 경로와 학습 설정

여기서는 `tb_master_exo.parquet`를 주 데이터 소스로 사용합니다.
다만 이 테이블에는 `demand_qty`가 없으므로, 학습 target은 `tb_master_target.parquet`에서 join합니다.

`MAX_IDS`를 줄이면 notebook 실행 속도를 빠르게 확인할 수 있고,
전체 데이터를 돌리고 싶으면 `MAX_IDS = None`으로 바꾸면 됩니다.


In [ ]:
DATA_ROOT = REPO_ROOT / "raw_data" / "master"
EXO_SOURCE = DATA_ROOT / "tb_master_exo.parquet"
TARGET_SOURCE = DATA_ROOT / "tb_master_target.parquet"

ID_COL = "oper_part_no"
DATE_COL = "demand_dt"
Y_COL = "demand_qty"
FREQ = "weekly"

# TimeXer는 non-overlap patch를 쓰므로 lookback이 patch_len으로 나누어떨어져야 합니다.
LOOKBACK = 52
HORIZON = 27
PATCH_LEN = 13

# notebook smoke run을 위해 series 수를 제한합니다. 전체 학습은 None으로 바꾸면 됩니다.
MAX_IDS = 32
VAL_RATIO = 0.2
BATCH_SIZE = 16

TRAIN_EPOCHS = 1
TRAIN_LR = 1e-3
TRAIN_DEVICE = DEFAULT_DEVICE

# TimeXer v1은 past continuous exogenous만 받습니다.
# part_prefix_2는 string이라 제외하고, part_group_id는 현재 v1에서 categorical path를 쓰지 않으므로 제외합니다.
PAST_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "part_len",
    "week_of_year",
    "promo_flag",
    "supply_outage_flag",
    "peak_season_flag",
    "is_year_start",
    "is_year_end",
    "is_q_start",
    "is_q_end",
]
FUTURE_EXO_CONT_COLS: list[str] = []
PAST_EXO_CAT_COLS: list[str] = []
EXCLUDED_EXO_COLS = ["seq", "part_prefix_2", "part_group_id"]

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "model_test" / "timexer_ab_dsio"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

config_snapshot = {
    "EXO_SOURCE": str(EXO_SOURCE),
    "TARGET_SOURCE": str(TARGET_SOURCE),
    "ID_COL": ID_COL,
    "DATE_COL": DATE_COL,
    "Y_COL": Y_COL,
    "FREQ": FREQ,
    "LOOKBACK": LOOKBACK,
    "HORIZON": HORIZON,
    "PATCH_LEN": PATCH_LEN,
    "MAX_IDS": MAX_IDS,
    "BATCH_SIZE": BATCH_SIZE,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "TRAIN_LR": TRAIN_LR,
    "TRAIN_DEVICE": TRAIN_DEVICE,
    "PAST_EXO_CONT_COLS": PAST_EXO_CONT_COLS,
    "ARTIFACT_DIR": str(ARTIFACT_DIR),
}
print(json.dumps(config_snapshot, indent=2, ensure_ascii=False))


## 2. Helper Functions

이 셀은 실제 TimeXer 학습에 필요한 공통 helper를 정의합니다.

- parquet 로딩
- 필수 컬럼 검증
- history가 충분한 id만 샘플링
- `tb_master_exo` + `tb_master_target`를 합쳐 one-table 학습 데이터 구성
- batch shape 출력
- `DataRequest` 생성


In [ ]:
def load_polars_table(source: Path, table_name: str) -> pl.DataFrame:
    if not source.exists():
        raise FileNotFoundError(f"{table_name} source not found: {source}")

    if source.suffix.lower() == ".parquet":
        return pl.read_parquet(source)
    if source.suffix.lower() in {".csv", ".txt"}:
        return pl.read_csv(source)

    raise ValueError(f"Unsupported file type for {table_name}: {source}")


def assert_columns(df: pl.DataFrame, required: list[str], table_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def sample_ids_by_observed_target(
    df: pl.DataFrame,
    *,
    id_col: str,
    y_col: str,
    min_obs: int,
    max_ids: int | None,
) -> list[str]:
    counts = (
        df.group_by(id_col)
        .agg(pl.col(y_col).is_not_null().sum().alias("observed_target_rows"))
        .filter(pl.col("observed_target_rows") >= min_obs)
        .sort("observed_target_rows", descending=True)
    )

    if max_ids is not None:
        counts = counts.head(max_ids)

    ids = counts[id_col].cast(pl.String).to_list()
    if not ids:
        raise ValueError(
            "No ids have enough observed target history. "
            f"Need at least {min_obs} non-null `{y_col}` rows per id."
        )
    return ids


def prepare_target_df(target_df: pl.DataFrame) -> pl.DataFrame:
    # target table은 TimeXer 학습용 y를 제공하는 역할만 합니다.
    assert_columns(target_df, [ID_COL, DATE_COL, Y_COL], "tb_master_target")

    return (
        target_df.select([ID_COL, DATE_COL, Y_COL])
        .drop_nulls([ID_COL, DATE_COL, Y_COL])
        .with_columns(
            pl.col(ID_COL).cast(pl.String),
            pl.col(DATE_COL).cast(pl.Int64),
            pl.col(Y_COL).cast(pl.Float64),
        )
        .sort([ID_COL, DATE_COL])
    )


def prepare_timexer_one_table(
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    *,
    lookback: int,
    horizon: int,
    max_ids: int | None,
) -> pl.DataFrame:
    # TimeXer window를 만들 수 있을 만큼 target 관측치가 있는 id만 남깁니다.
    min_obs = lookback + horizon
    keep_ids = sample_ids_by_observed_target(
        target_df,
        id_col=ID_COL,
        y_col=Y_COL,
        min_obs=min_obs,
        max_ids=max_ids,
    )

    assert_columns(exo_df, [ID_COL, DATE_COL, *PAST_EXO_CONT_COLS], "tb_master_exo")

    joined = (
        exo_df.join(target_df, on=[ID_COL, DATE_COL], how="left")
        .filter(pl.col(ID_COL).cast(pl.String).is_in(keep_ids))
    )

    # supervised training에서는 target이 없는 row를 쓰지 않으므로 제거합니다.
    joined = joined.filter(pl.col(Y_COL).is_not_null())

    selected_cols = [ID_COL, DATE_COL, Y_COL, *PAST_EXO_CONT_COLS]
    return (
        joined.select(selected_cols)
        .with_columns(
            pl.col(ID_COL).cast(pl.String),
            pl.col(DATE_COL).cast(pl.Int64),
            pl.col(Y_COL).cast(pl.Float64),
            *[pl.col(col).cast(pl.Float64) for col in PAST_EXO_CONT_COLS],
        )
        .sort([ID_COL, DATE_COL])
    )


def describe_batch(batch, label: str) -> None:
    names = ["x", "y", "uid_list", "future_exo", "past_exo_cont", "past_exo_cat"]
    print(f"=== {label} batch ===")
    for name, item in zip(names, batch):
        if torch.is_tensor(item):
            print(f"{name:14s}: shape={tuple(item.shape)}, dtype={item.dtype}")
        elif isinstance(item, list):
            print(f"{name:14s}: list(len={len(item)})")
        else:
            print(f"{name:14s}: {type(item).__name__}")


def make_data_request(df: pl.DataFrame, *, stage: str) -> DataRequest:
    # TimeXer v1은 future exo를 쓰지 않으므로 future_exo_cont_cols는 빈 리스트로 둡니다.
    return DataRequest(
        df=df,
        window=DataWindowConfig(lookback=LOOKBACK, horizon=HORIZON, freq=FREQ),
        columns=DataColumnConfig(id_col=ID_COL, date_col=DATE_COL, y_col=Y_COL),
        exogenous=ExogenousConfig(
            use_exogenous_mode=True,
            past_exo_cont_cols=PAST_EXO_CONT_COLS,
            past_exo_cat_cols=PAST_EXO_CAT_COLS,
            future_exo_cont_cols=FUTURE_EXO_CONT_COLS,
        ),
        loader=LoaderConfig(
            batch_size=BATCH_SIZE,
            val_ratio=VAL_RATIO,
            shuffle=(stage == "train"),
            seed=SEED,
            stage=stage,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=False,
        ),
    )


## 3. 데이터 로딩과 one-table 학습 데이터 구성

이 단계에서는 아래 순서로 데이터를 준비합니다.

- `tb_master_target`에서 target history를 읽습니다.
- `lookback + horizon` 이상 history가 있는 id만 고릅니다.
- `tb_master_exo`에 `demand_qty`를 join합니다.
- TimeXer가 실제로 쓸 컬럼만 남겨서 학습용 long table을 만듭니다.


In [ ]:
target_raw = load_polars_table(TARGET_SOURCE, "tb_master_target")
exo_raw = load_polars_table(EXO_SOURCE, "tb_master_exo")

target_df = prepare_target_df(target_raw)
train_df = prepare_timexer_one_table(
    target_df,
    exo_raw,
    lookback=LOOKBACK,
    horizon=HORIZON,
    max_ids=MAX_IDS,
)

available_exo_cols = [
    col
    for col in exo_raw.columns
    if col not in {ID_COL, DATE_COL, *EXCLUDED_EXO_COLS}
]
missing_exo_cols = [col for col in PAST_EXO_CONT_COLS if col not in exo_raw.columns]
if missing_exo_cols:
    raise ValueError(f"Missing TimeXer exogenous columns: {missing_exo_cols}")

print("target_raw shape:", target_raw.shape)
print("exo_raw shape   :", exo_raw.shape)
print("train_df shape  :", train_df.shape)
print("sample id count :", train_df[ID_COL].n_unique())
print("available exo columns:")
print(available_exo_cols)

train_df.head(10)


## 4. Dataloader 생성과 batch shape 확인

학습 전에 batch shape를 먼저 확인해두면,
TimeXer 입력이 정말로 `x + past_exo_cont` 형태로 잡혔는지 빠르게 검증할 수 있습니다.

여기서 기대하는 포인트는 다음과 같습니다.
- `x`: `(B, LOOKBACK, 1)`
- `future_exo`: `(B, HORIZON, 0)`
- `past_exo_cont`: `(B, LOOKBACK, len(PAST_EXO_CONT_COLS))`


In [ ]:
train_data_request = make_data_request(train_df, stage="train")
val_data_request = make_data_request(train_df, stage="val")

train_loader = build_dataloader(train_data_request)
val_loader = build_dataloader(val_data_request)

train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))

describe_batch(train_batch, "train")
describe_batch(val_batch, "val")


## 5. TimeXer 학습 요청 구성

여기서는 `timexer_base` 하나만 학습합니다.

중요한 포인트는 다음과 같습니다.
- `use_exogenous_mode=True`
- `future_exo_cont_cols=[]`
- `patch_len=13`으로 지정해서 `lookback=52`와 맞춤
- 현재 public `train()` API는 `data.window` 외에도 top-level `freq/lookback/horizon`를 같이 주는 편이 안전함


In [ ]:
train_request = TrainRequest(
    models=["timexer_base"],
    data=train_data_request,
    # 현재 train() API가 top-level window 정보도 읽기 때문에 같이 넣어 둡니다.
    lookback=LOOKBACK,
    horizon=HORIZON,
    freq=FREQ,
    use_exogenous_mode=True,
    trainer=TrainerConfig(epochs=TRAIN_EPOCHS, lr=TRAIN_LR),
    runtime=RuntimeConfig(device=TRAIN_DEVICE),
    artifacts=ArtifactConfig(save_dir=str(ARTIFACT_DIR), auto_save_dir=False),
    architecture=ArchitectureConfig(
        timexer=TimexerArchitectureConfig(
            patch_len=PATCH_LEN,
            d_model=64,
            n_heads=4,
            d_ff=128,
            e_layers=1,
            dropout=0.1,
            activation="gelu",
            use_norm=True,
        )
    ),
)

train_request


## 6. 학습 실행과 결과 확인

이 셀을 실행하면 실제 `timexer_base` 학습이 시작됩니다.
완료 후에는 아래 정보를 확인합니다.

- 요청한 모델 목록
- checkpoint 경로
- training manifest 경로
- 요약 결과 dict


In [ ]:
train_result = train(train_request)

print("requested_models:", train_result.requested_models)
print("save_dir       :", train_result.save_dir)
print("manifest_path  :", train_result.manifest_path)
print("ckpt_paths     :")
print(json.dumps(train_result.ckpt_paths, indent=2, ensure_ascii=False))
print("results        :")
pprint(train_result.results)


## 다음 조정 포인트

- 전체 데이터를 돌리고 싶으면 `MAX_IDS = None`으로 변경
- 학습을 더 길게 보고 싶으면 `TRAIN_EPOCHS` 증가
- 모델 크기를 키우고 싶으면 `d_model / n_heads / d_ff / e_layers` 조정
- `part_group_id` 같은 categorical feature를 쓰고 싶으면, 먼저 연속형으로 인코딩한 뒤 `PAST_EXO_CONT_COLS`로 넣기
